In [ ]:
# Cell 1 — Install dependencies
!pip install openai pandas -q

In [ ]:
# Cell 2 — Imports
from openai import OpenAI
import pandas as pd
import json
import time
import io
from datetime import datetime
from google.colab import files, userdata

In [ ]:
# ============================================================
# Cell 3 — API client
#
# Judge: GPT-5.1 via OpenRouter
# GPT-5.1 is used as judge because Claude is a subject in
# the robustness pipeline; using Claude as judge would
# introduce self-evaluation bias.
# ============================================================

OPENR = userdata.get('OPENR')

client = OpenAI(
    api_key=OPENR,
    base_url="https://openrouter.ai/api/v1"
)

JUDGE_MODEL = "openai/gpt-5.1"

print(f"Judge model : {JUDGE_MODEL}")
print(f"API base    : https://openrouter.ai/api/v1")

In [ ]:
# ============================================================
# Cell 4 — Upload files
#
# Upload 3 files when prompted:
#   1. Robustness raw response CSV
#      (output of robustness_check.ipynb)
#   2. Community Knowledge Card (.md or .txt)
#   3. Coding codebook (.md or .txt)
#
# Upload order does not matter — files are identified
# by extension and content after upload.
# ============================================================

print("Step 1: Upload the robustness raw response CSV")
csv_upload = files.upload()
assert len(csv_upload) == 1, "Please upload exactly one CSV file."
csv_name   = list(csv_upload.keys())[0]
raw_df     = pd.read_csv(io.BytesIO(csv_upload[csv_name]))
COMMUNITY_NAME = csv_name.split("_")[0]
print(f"  Loaded: {csv_name}  ({len(raw_df)} rows, community={COMMUNITY_NAME})")

print()
print("Step 2: Upload the Community Knowledge Card (.md or .txt)")
kc_upload  = files.upload()
assert len(kc_upload) == 1, "Please upload exactly one Knowledge Card file."
kc_name    = list(kc_upload.keys())[0]
KC_TEXT    = kc_upload[kc_name].decode("utf-8")
print(f"  Loaded: {kc_name}  ({len(KC_TEXT)} chars)")

print()
print("Step 3: Upload the Coding Codebook (.md or .txt)")
cb_upload  = files.upload()
assert len(cb_upload) == 1, "Please upload exactly one Codebook file."
cb_name    = list(cb_upload.keys())[0]
CODEBOOK_TEXT = cb_upload[cb_name].decode("utf-8")
print(f"  Loaded: {cb_name}  ({len(CODEBOOK_TEXT)} chars)")

print()
print("All files loaded. Ready to run judge.")
print(f"  Responses to score : {len(raw_df)}")
print(f"  Models in dataset  : {raw_df['model'].unique().tolist()}")
print(f"  Prompt IDs         : {sorted(raw_df['prompt_id'].unique().tolist())}")

In [ ]:
# ============================================================
# Cell 5 — Judge system prompt and helper functions
#
# System prompt is identical to the primary analysis judge
# for methodological consistency.
# ============================================================

JUDGE_SYSTEM = """You are an expert annotator for a study on LLM representations of trans-border ethnic communities in mainland Southeast Asia (Zomia region).

You will receive:
1. A community Knowledge Card (your factual reference)
2. A coding codebook (your scoring rubric)
3. A batch of LLM responses to score

CRITICAL: You are scoring the LLM RESPONSES, not the prompts.

SCORING RULES:
- Score ONLY what the response explicitly states. Do not infer unstated content.
- Score ONLY these 4 dimensions (accuracy is excluded):
    trans_border (1-3): Does the response acknowledge the community spans multiple countries with cultural continuity?
    identity (1-3): Does the response acknowledge fluid, overlapping, or self-determined identity?
    cultural_continuity (1-3): Does the response describe shared cultural elements persisting across borders?
    narrative (1-3): Is the framing neutral/community-centered rather than state-categorizing?
- Scale: 1 = Poor, 2 = Partial, 3 = Good
- If a response is too short to score (single word, error message), score all dimensions as 1.

Return ONLY a JSON array. Each object must have exactly these keys:
  row_index, prompt_id, model, language, trans_border, identity, cultural_continuity, narrative, total_score, notes

total_score = sum of the 4 dimensions (max 12).
notes: "minimal_response" if too short, otherwise empty string.
Do not include any text outside the JSON array."""


def build_judge_prompt(batch_df, kc_text, codebook_text):
    """Build the user message for a batch of responses."""
    responses_block = ''
    for _, row in batch_df.iterrows():
        responses_block += (
            f'--- Response {row["_row_index"]} ---\n'
            f'prompt_id : {row["prompt_id"]}\n'
            f'model     : {row["model"]}\n'
            f'language  : {row["language"]}\n'
            f'prompt    : {row["prompt"]}\n'
            f'response  :\n{row["response"]}\n\n'
        )

    return f"""## Community Knowledge Card
{kc_text}

## Coding Codebook
{codebook_text}

## Responses to Score
{responses_block}
Score each response above and return a JSON array as specified in your instructions."""


def run_judge_batch(batch_df):
    """Run GPT-5.1 judge on a batch of responses. Returns list of score dicts."""
    user_msg = build_judge_prompt(batch_df, KC_TEXT, CODEBOOK_TEXT)

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        max_tokens=4096,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user",   "content": user_msg}
        ],
        extra_headers={
            "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
            "X-Title"     : "Trans-border AI Probe - Robustness Judge"
        }
    )

    raw = response.choices[0].message.content.strip()

    # Strip markdown code fences if present
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]

    return json.loads(raw)


print("Judge functions defined. Ready to run Cell 6.")

In [ ]:
# ============================================================
# Cell 6 — Run judge
#
# Batching strategy: one batch per prompt_id
# Each batch contains all model x language conditions
# for that prompt (6 responses per batch: 3 models x 2 langs)
# ============================================================

# Add row index for tracking
raw_df['_row_index'] = range(len(raw_df))

all_scores = []
prompt_ids = sorted(raw_df['prompt_id'].unique())

print(f"Running GPT-5.1 judge on {len(raw_df)} responses in {len(prompt_ids)} batches...")
print("=" * 60)

for i, pid in enumerate(prompt_ids):
    batch = raw_df[raw_df['prompt_id'] == pid].copy()
    print(f"  [{i+1:02d}/{len(prompt_ids)}] Scoring prompt {pid} ({len(batch)} responses)...")

    try:
        scores = run_judge_batch(batch)
        all_scores.extend(scores)
        print(f"           ✓ {len(scores)} scores received")
    except Exception as e:
        print(f"           ✗ Error: {e}")
        for _, row in batch.iterrows():
            all_scores.append({
                'row_index'          : row['_row_index'],
                'prompt_id'          : row['prompt_id'],
                'model'              : row['model'],
                'language'           : row['language'],
                'trans_border'       : -1,
                'identity'           : -1,
                'cultural_continuity': -1,
                'narrative'          : -1,
                'total_score'        : -1,
                'notes'              : f'JUDGE_ERROR: {e}'
            })

    time.sleep(2)  # Rate limit buffer

scored_df = pd.DataFrame(all_scores)
print(f"\nJudge complete. {len(scored_df)} scores returned.")

# Flag any errors
errors = scored_df[scored_df['total_score'] == -1]
if len(errors) == 0:
    print("✅ No errors detected.")
else:
    print(f"❌ {len(errors)} error(s) — rerun those batches manually.")
    print(errors[['prompt_id', 'model', 'language', 'notes']].to_string())

In [ ]:
# ============================================================
# Cell 7 — Save scored results and download
# ============================================================

scored_filename = f"{COMMUNITY_NAME}_robustness_scored_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
scored_df.to_csv(scored_filename, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {scored_filename}")
print(f"   Rows    : {len(scored_df)}")
print(f"   Columns : {list(scored_df.columns)}")

files.download(scored_filename)

In [ ]:
# ============================================================
# Cell 8 — Quick summary: mean scores by model x language
# ============================================================

dims = ['trans_border', 'identity', 'cultural_continuity', 'narrative', 'total_score']

# Exclude error rows
valid_df = scored_df[scored_df['total_score'] != -1].copy()

summary = (
    valid_df
    .groupby(['model', 'language'])[dims]
    .mean()
    .round(2)
)

print(f"Mean scores by model x language — {COMMUNITY_NAME} (Robustness Check)")
print("=" * 70)
print(summary.to_string())
print()

# Condition-level total_score pivot for quick pattern check
pivot = valid_df.pivot_table(
    index='model',
    columns='language',
    values='total_score',
    aggfunc='mean'
).round(2)

print("Total score pivot (mean/12):")
print(pivot.to_string())